[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/ia-startups/03-costes-y-escalado.ipynb)

# Costes y Escalado de IA

Monitor de costes, router de modelos y optimización para startups en crecimiento.

In [ ]:
import anthropic
import json
import time
from collections import defaultdict
from datetime import date

client = anthropic.Anthropic()

PRECIOS = {
    "claude-haiku-4-5-20251001": {"input": 0.80, "output": 4.00},
    "claude-sonnet-4-6": {"input": 3.00, "output": 15.00},
}

print("Monitor de costes inicializado")

## 1. Monitor de costes por usuario

In [ ]:
class MonitorCostes:
    def __init__(self):
        self._costes: dict = defaultdict(lambda: defaultdict(float))
        self._llamadas: dict = defaultdict(int)
        self.limites_diarios: dict = {}

    def registrar(self, uid: str, modelo: str, tok_in: int, tok_out: int) -> dict:
        p = PRECIOS.get(modelo, PRECIOS["claude-haiku-4-5-20251001"])
        coste = (tok_in * p["input"] + tok_out * p["output"]) / 1_000_000
        hoy = date.today().isoformat()
        self._costes[uid][hoy] += coste
        self._llamadas[uid] += 1
        limite = self.limites_diarios.get(uid, 2.0)
        coste_hoy = self._costes[uid][hoy]
        return {
            "coste_llamada_usd": round(coste, 5),
            "coste_hoy_usd": round(coste_hoy, 4),
            "pct_limite": round(coste_hoy / limite * 100, 1),
            "bloqueado": coste_hoy >= limite
        }

    def puede_llamar(self, uid: str) -> bool:
        hoy = date.today().isoformat()
        return self._costes[uid][hoy] < self.limites_diarios.get(uid, 2.0)

    def resumen_cartera(self) -> list:
        hoy = date.today().isoformat()
        return sorted([
            {"uid": uid, "coste_hoy": round(dias.get(hoy, 0), 4),
             "llamadas": self._llamadas[uid],
             "coste_total": round(sum(dias.values()), 4)}
            for uid, dias in self._costes.items()
        ], key=lambda x: x["coste_total"], reverse=True)


monitor = MonitorCostes()
monitor.limites_diarios = {"user_free": 0.10, "user_pro": 1.00, "user_biz": 5.00}

# Simular uso de distintos usuarios
import random
random.seed(42)

usuarios = [("user_free", 3), ("user_pro", 15), ("user_biz", 50)]
for uid, n_llamadas in usuarios:
    for _ in range(n_llamadas):
        tok_in = random.randint(500, 1500)
        tok_out = random.randint(200, 600)
        m = monitor.registrar(uid, "claude-haiku-4-5-20251001", tok_in, tok_out)

print("RESUMEN DE CARTERA")
print(f"{'Usuario':<12} {'Llamadas':>10} {'Coste hoy':>12} {'Coste total':>12}")
print("-" * 50)
for r in monitor.resumen_cartera():
    limite = monitor.limites_diarios.get(r['uid'], 2.0)
    pct = round(r['coste_hoy'] / limite * 100)
    print(f"{r['uid']:<12} {r['llamadas']:>10} ${r['coste_hoy']:>10.4f} ${r['coste_total']:>10.4f}  ({pct}% límite)")

## 2. Router de modelos por complejidad

In [ ]:
PALABRAS_SONNET = [
    "analiza", "razona", "compara", "evalúa", "estrategia",
    "diseña", "argumenta", "sintetiza", "critica", "implicaciones"
]

def elegir_modelo(mensaje: str, historial_len: int = 0) -> tuple[str, str]:
    """Devuelve (modelo_id, motivo)."""
    longitud = len(mensaje)
    es_complejo = any(p in mensaje.lower() for p in PALABRAS_SONNET)
    historial_largo = historial_len > 6

    if es_complejo or longitud > 800 or historial_largo:
        return "claude-sonnet-4-6", "tarea compleja detectada"
    return "claude-haiku-4-5-20251001", "tarea simple — coste mínimo"


def chat_enrutado(uid: str, mensaje: str, historial: list = None) -> dict:
    historial = historial or []
    if not monitor.puede_llamar(uid):
        return {"error": "Límite diario alcanzado", "modelo": None}

    modelo, motivo = elegir_modelo(mensaje, len(historial))
    inicio = time.perf_counter()

    resp = client.messages.create(
        model=modelo,
        max_tokens=600,
        messages=historial + [{"role": "user", "content": mensaje}]
    )
    latencia = round((time.perf_counter() - inicio) * 1000)
    coste_info = monitor.registrar(uid, modelo, resp.usage.input_tokens, resp.usage.output_tokens)

    modelo_label = "Sonnet" if "sonnet" in modelo else "Haiku"
    return {
        "respuesta": resp.content[0].text,
        "modelo": modelo_label,
        "motivo": motivo,
        "latencia_ms": latencia,
        "coste_usd": coste_info["coste_llamada_usd"]
    }


CONSULTAS_DEMO = [
    "¿Hola, cómo funciona esto?",
    "Analiza las implicaciones estratégicas de la limitación de responsabilidad en contratos SaaS B2B",
    "¿Qué significa NDA?",
    "Compara y argumenta las diferencias entre arbitraje y mediación como métodos de resolución de disputas",
    "¿Puedo cancelar el contrato?",
]

print("DEMO DEL ROUTER DE MODELOS")
print(f"{'Consulta':<55} {'Modelo':>8} {'Coste':>10} {'Latencia':>10}")
print("-" * 90)

for consulta in CONSULTAS_DEMO:
    r = chat_enrutado("user_pro", consulta)
    print(f"{consulta[:53]:<55} {r['modelo']:>8} ${r['coste_usd']:>8.5f} {r['latencia_ms']:>8}ms")

## 3. Prompt Caching para system prompts largos

In [ ]:
# System prompt largo con base de conocimiento legal embebida
SYSTEM_CON_BASE_CONOCIMIENTO = """Eres un asistente de análisis de contratos para PYMEs españolas.
Identifica cláusulas problemáticas, indica el riesgo (BAJO/MEDIO/ALTO) y sugiere alternativas.

BASE DE CONOCIMIENTO LEGAL ESPAÑOLA:
- Código Civil: Art. 1091 (contratos tienen fuerza de ley entre las partes)
- Código Civil: Art. 1255 (libertad de pacto, con límites de ley, moral y orden público)
- LGDCU: Art. 82-90 (cláusulas abusivas en contratos B2C, también referencia en B2B)
- ET: Art. 10-11 (contratos de trabajo temporales y sus límites)
- LOPD/GDPR: Tratamiento de datos en contratos de servicios
- Ley de Competencia Desleal: Cláusulas que restringen competencia post-contractual
- Incoterms 2020: Términos de comercio internacional si aplica
- Ley de Morosidad (Ley 3/2004): Plazos de pago B2B máximos (60 días)
- Reglamento Roma I: Ley aplicable en contratos internacionales
- Convenio de Bruselas: Jurisdicción en disputas internacionales

PATRONES DE RIESGO IDENTIFICADOS EN 4.200+ CONTRATOS:
- Limitación de responsabilidad total: SIEMPRE ALTO RIESGO
- Renovación automática sin notificación adecuada (< 30 días): ALTO RIESGO
- Jurisdicción fuera de España para empresa española: ALTO RIESGO
- Propiedad intelectual cedida sin compensación: ALTO RIESGO
- Cláusula de no competencia desproporcionada: MEDIO-ALTO RIESGO
- Subcontratación sin consentimiento previo: MEDIO RIESGO
- Indexación al IPC: BAJO RIESGO (estándar del mercado)

REGLAS OBLIGATORIAS:
- Nunca des asesoramiento jurídico vinculante
- Siempre sugiere consultar con abogado para decisiones importantes
- Cita el artículo legal relevante cuando sea útil
- Sé conciso: máximo 3 puntos por cláusula"""

def llamar_con_cache(mensaje: str, n_llamada: int) -> dict:
    """Llama con cache_control en el system prompt largo."""
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        system=[
            {
                "type": "text",
                "text": SYSTEM_CON_BASE_CONOCIMIENTO,
                "cache_control": {"type": "ephemeral"}
            }
        ],
        messages=[{"role": "user", "content": mensaje}]
    )

    write = resp.usage.cache_creation_input_tokens
    read = resp.usage.cache_read_input_tokens
    normal = resp.usage.input_tokens

    # Coste sin caché
    sin_cache = (normal + write + read) * 0.80 / 1_000_000
    # Coste con caché
    con_cache = (normal * 0.80 + write * 1.00 + read * 0.08) / 1_000_000
    ahorro = max(0, 1 - con_cache / sin_cache) * 100 if sin_cache > 0 else 0

    return {
        "llamada": n_llamada,
        "cache_write": write,
        "cache_read": read,
        "coste_sin_cache": round(sin_cache, 5),
        "coste_con_cache": round(con_cache, 5),
        "ahorro_pct": round(ahorro, 1)
    }


print(f"System prompt: {len(SYSTEM_CON_BASE_CONOCIMIENTO)} chars (~{len(SYSTEM_CON_BASE_CONOCIMIENTO)//4} tokens)")
print("\nEfecto del caché en llamadas sucesivas:")
print(f"{'Llamada':>8} {'Cache Write':>12} {'Cache Read':>12} {'Sin caché':>12} {'Con caché':>12} {'Ahorro':>8}")
print("-" * 70)

mensajes = [
    "Analiza: El proveedor no asume responsabilidad por ningún daño.",
    "Analiza: La renovación es automática cada año sin previo aviso.",
    "Analiza: La jurisdicción será la de los tribunales de Londres.",
]
for i, msg in enumerate(mensajes, 1):
    r = llamar_con_cache(msg, i)
    tipo = "[WRITE]" if r["cache_write"] > 0 else "[READ] "
    print(f"{tipo} #{r['llamada']:>3}   {r['cache_write']:>10}   {r['cache_read']:>10}   ${r['coste_sin_cache']:>9.5f}   ${r['coste_con_cache']:>9.5f}   {r['ahorro_pct']:>6.1f}%")

## 4. Proyección de costes a escala

In [ ]:
def proyectar_costes(usuarios: int, analisis_por_usuario_mes: int,
                     tokens_por_analisis: int = 1200,
                     pct_cache_hit: float = 0.70) -> dict:
    """Proyecta costes mensuales con y sin optimizaciones."""

    tokens_totales_mes = usuarios * analisis_por_usuario_mes * tokens_por_analisis

    # Sin optimización: todo Haiku, sin caché
    coste_base = tokens_totales_mes / 1_000_000 * 2.40  # mixto input/output

    # Con router: 80% Haiku, 20% Sonnet
    coste_router = tokens_totales_mes / 1_000_000 * (0.80 * 2.40 + 0.20 * 9.00)

    # Con caché: 70% cache_read (90% más barato), 30% normal
    tokens_cache = tokens_totales_mes * pct_cache_hit
    tokens_normal = tokens_totales_mes * (1 - pct_cache_hit)
    coste_cache = (tokens_normal * 2.40 + tokens_cache * 0.24) / 1_000_000

    # Combinado: router + caché
    coste_optimo = coste_cache * 0.85  # estimación combinada

    return {
        "usuarios": usuarios,
        "analisis_mes": usuarios * analisis_por_usuario_mes,
        "coste_base_usd": round(coste_base, 2),
        "coste_con_router_usd": round(coste_router, 2),
        "coste_con_cache_usd": round(coste_cache, 2),
        "coste_optimo_usd": round(coste_optimo, 2),
        "ahorro_vs_base_pct": round((1 - coste_optimo / coste_base) * 100, 1)
    }


print("PROYECCIÓN DE COSTES MENSUAL")
print(f"{'Usuarios':>10} {'Análisis':>10} {'Base':>10} {'Con Caché':>12} {'Óptimo':>10} {'Ahorro':>8}")
print("-" * 65)

for n_usuarios in [50, 200, 500, 1000, 5000]:
    p = proyectar_costes(n_usuarios, analisis_por_usuario_mes=30)
    print(f"{p['usuarios']:>10,} {p['analisis_mes']:>10,} ${p['coste_base_usd']:>8,.2f} ${p['coste_con_cache_usd']:>10,.2f} ${p['coste_optimo_usd']:>8,.2f} {p['ahorro_vs_base_pct']:>7.1f}%")